# Dukascopy EUR/USD Tick Ingest

Downloads, validates, and stores Dukascopy tick data as Parquet.

**Date range**: `START_DATE` / `END_DATE` in the config cell.

**Output**
- Daily: `EURUSD_YYYY-MM-DD_ticks_utc.parquet` (removed after merge when enabled)
- Combined: `EURUSD_ticks_utc_combined_YYYYMMDD_YYYYMMDD.parquet`
- `manifest.csv`, `validation_report.csv`

**Validation report** = one row per problem only; empty body is normal.


In [1]:
from __future__ import annotations

import json
import lzma
import re
import struct
import time
from datetime import datetime, timedelta, timezone
from pathlib import Path
from urllib.request import Request, urlopen

import pandas as pd
import pyarrow.parquet as pq


In [2]:
# --- DATE RANGE (edit END_DATE to extend) --------------------------------
START_DATE = "2021-03-01"
END_DATE   = "2026-03-27"  # Friday

# After ingest: merge all daily Parquet into one file, then delete dailies
MERGE_TO_SINGLE_PARQUET = True
DELETE_DAILY_AFTER_MERGE = True

# --- SYMBOL -------------------------------------------------------------
SYMBOL = "EURUSD"
PRICE_SCALE = 1e5

# --- HTTP (fewer false "missing" hours when the feed is flaky) ---
HTTP_TIMEOUT_SEC = 60
HTTP_RETRIES = 5
HTTP_RETRY_BACKOFF_SEC = 0.8
DELAY_BETWEEN_HOURS_SEC = 0.05  # set 0 to disable pause between hours

# --- VALIDATION THRESHOLDS ----------------------------------------------
PRICE_MIN      = 0.5
PRICE_MAX      = 2.5
SPREAD_MAX_OK  = 0.005
SPREAD_MAX_ERR = 0.020

# --- PATHS --------------------------------------------------------------
def _repo_root() -> Path:
    cur = Path.cwd().resolve()
    for p in [cur, *cur.parents]:
        if (p / "data").exists() and (p / "src").exists():
            return p
    raise RuntimeError("Cannot locate project root.")

ROOT        = _repo_root()
OUT_DIR     = ROOT / "data" / "dirty" / "ticks" / "raw_source" / "dukascopy"
CAL_PATH    = ROOT / "data" / "dirty" / "calendar" / "us_calendar.parquet"
MANIFEST_PATH = OUT_DIR / "manifest.csv"
REPORT_PATH   = OUT_DIR / "validation_report.csv"

OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Range  : {START_DATE} -> {END_DATE}")
print(f"Output : {OUT_DIR}")
print(f"Merge  : {MERGE_TO_SINGLE_PARQUET}  delete dailies: {DELETE_DAILY_AFTER_MERGE}")


Range  : 2021-03-01 -> 2026-03-27
Output : C:\Users\user\EUR_USD_Project\data\dirty\ticks\raw_source\dukascopy
Merge  : True  delete dailies: True


In [3]:
cal = pd.read_parquet(CAL_PATH)
cal["date"] = pd.to_datetime(cal["date"]).dt.normalize()
cal = cal.set_index("date")
print(f"Calendar loaded: {len(cal)} days")

Calendar loaded: 2557 days


In [4]:
def dukascopy_url(symbol: str, day: datetime, hour: int) -> str:
    """Build the Dukascopy bi5 URL. Month is zero-based in their scheme."""
    return (
        f"https://datafeed.dukascopy.com/datafeed/{symbol}/"
        f"{day.year}/{day.month - 1:02d}/{day.day:02d}/{hour:02d}h_ticks.bi5"
    )


def decode_bi5(blob: bytes, hour_start: datetime) -> pd.DataFrame:
    """Decompress and decode a single bi5 hour blob into a DataFrame."""
    raw = lzma.decompress(blob)
    rec_size = 20  # uint32 ms + uint32 ask + uint32 bid + float32 ask_vol + float32 bid_vol
    n = len(raw) // rec_size
    rows = []
    for i in range(n):
        ms, ask_i, bid_i, ask_vol, bid_vol = struct.unpack_from(">IIIff", raw, i * rec_size)
        ts = hour_start + timedelta(milliseconds=int(ms))
        rows.append((ts, bid_i / PRICE_SCALE, ask_i / PRICE_SCALE, ask_vol, bid_vol))
    return pd.DataFrame(rows, columns=["timestamp_utc", "bid", "ask", "bid_volume", "ask_volume"])


def fetch_bi5_hour(url: str) -> bytes | None:
    """GET one hour blob; retries on empty body / errors (Dukascopy can flake under bulk load)."""
    req = Request(
        url,
        headers={
            "User-Agent": (
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
            )
        },
    )
    backoff = HTTP_RETRY_BACKOFF_SEC
    for attempt in range(HTTP_RETRIES):
        try:
            with urlopen(req, timeout=HTTP_TIMEOUT_SEC) as resp:
                blob = resp.read()
            if blob:
                return blob
        except Exception:
            pass
        if attempt < HTTP_RETRIES - 1:
            time.sleep(backoff * (2 ** attempt))
    return None


def download_day(symbol: str, day_str: str) -> tuple[pd.DataFrame, list[int]]:
    """Download and decode all 24 hours for one day. Returns (df, missing_hours)."""
    day = datetime.strptime(day_str, "%Y-%m-%d").replace(tzinfo=timezone.utc)
    chunks: list[pd.DataFrame] = []
    missing_hours: list[int] = []

    for hour in range(24):
        hour_start = day + timedelta(hours=hour)
        url = dukascopy_url(symbol, day, hour)
        blob = fetch_bi5_hour(url)
        if not blob:
            missing_hours.append(hour)
            if DELAY_BETWEEN_HOURS_SEC:
                time.sleep(DELAY_BETWEEN_HOURS_SEC)
            continue

        try:
            df_hour = decode_bi5(blob, hour_start)
            chunks.append(df_hour)
        except Exception as exc:
            missing_hours.append(hour)
            print(f"  [WARN] decode error {day_str} h{hour:02d}: {exc}")

        if DELAY_BETWEEN_HOURS_SEC:
            time.sleep(DELAY_BETWEEN_HOURS_SEC)

    if not chunks:
        df = pd.DataFrame(columns=["timestamp_utc", "bid", "ask", "bid_volume", "ask_volume"])
    else:
        df = pd.concat(chunks, ignore_index=True)

    df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"], utc=True)
    return df, missing_hours


In [5]:
def validate_day(
    df: pd.DataFrame,
    day_str: str,
    missing_hours: list[int],
    cal: pd.DataFrame,
) -> tuple[str, list[str], list[dict]]:
    """
    Run all validation checks for one day.
    Returns (status, warnings_list, report_rows).
    status = 'ok' | 'warn' | 'error'
    """
    issues: list[dict] = []   # rows for validation_report.csv
    warn_codes: list[str] = []

    day_ts = pd.Timestamp(day_str)
    cal_row = cal.loc[day_ts] if day_ts in cal.index else None

    is_weekend      = bool(cal_row["is_weekend"])      if cal_row is not None else None
    is_us_holiday   = bool(cal_row["is_us_holiday"])   if cal_row is not None else None
    is_expected     = bool(cal_row["is_expected_data"]) if cal_row is not None else None
    day_of_week     = int(cal_row["day_of_week"])       if cal_row is not None else None  # 0=Mon â€¦ 6=Sun
    is_friday       = (day_of_week == 4)

    # Missing hours that are NOT explained by known FX close patterns
    friday_eod_hours = {21, 22, 23}
    missing_set = set(missing_hours)
    missing_hours_expected = (
        bool(is_weekend)
        or bool(is_us_holiday)
        or (is_friday and missing_set.issubset(friday_eod_hours))
    )

    def add(issue: str, detail: str, severity: str) -> None:
        warn_codes.append(issue)
        issues.append({"date": day_str, "issue": issue, "detail": detail, "severity": severity})

    rows = len(df)

    # 1. Calendar â€” empty when data expected (weekends and holidays are fine with 0 rows)
    if rows == 0 and is_expected and not is_weekend and not is_us_holiday:
        add("empty_expected_day", f"0 rows on expected-data day (weekend={is_weekend}, holiday={is_us_holiday})", "error")

    # 2. Missing hours â€” skip entirely if weekend, holiday, or Friday late close
    if len(missing_hours) > 0 and not missing_hours_expected:
        sev = "warn" if len(missing_hours) <= 6 else "error"
        add("missing_hours", f"{len(missing_hours)} hours missing: {missing_hours}", sev)

    if rows == 0:
        if any(i["severity"] == "error" for i in issues):
            status = "error"
        elif issues:
            status = "warn"
        else:
            status = "ok"
        return status, warn_codes, issues

    day_start = pd.Timestamp(day_str, tz="UTC")
    day_end   = day_start + pd.Timedelta(days=1)

    # 3. min_ts / max_ts inside intended UTC day
    min_ts = df["timestamp_utc"].min()
    max_ts = df["timestamp_utc"].max()
    if min_ts < day_start or max_ts >= day_end:
        add("ts_out_of_day", f"min={min_ts}, max={max_ts}, expected [{day_start}, {day_end})", "error")

    # 4. No timestamps outside the day
    outside = ((df["timestamp_utc"] < day_start) | (df["timestamp_utc"] >= day_end)).sum()
    if outside > 0:
        add("timestamps_outside_day", f"{outside} rows outside [{day_start}, {day_end})", "error")

    # 5. Sort check â€” non-decreasing
    if not df["timestamp_utc"].is_monotonic_increasing:
        add("unsorted_timestamps", "timestamp_utc is not non-decreasing", "error")

    # 6. Duplicate timestamps
    dup_ts = df["timestamp_utc"].duplicated().sum()
    if dup_ts > 100:
        add("high_dup_ts_count", f"dup_ts_count={dup_ts}", "warn")

    # 7. Bad values â€” NaN / inf
    for col in ["bid", "ask"]:
        bad = (~df[col].apply(lambda x: pd.notna(x) and x not in (float("inf"), float("-inf")))).sum()
        if bad > 0:
            add("bad_values", f"{bad} NaN/inf in column '{col}'", "error")

    # 8. bid <= ask
    bid_gt_ask = (df["bid"] > df["ask"]).sum()
    if bid_gt_ask > 0:
        add("bid_gt_ask", f"{bid_gt_ask} rows where bid > ask", "error")

    # 8b. Volumes: finite; 0 is OK; negative is not
    for col in ("bid_volume", "ask_volume"):
        bad = (~df[col].apply(lambda x: pd.notna(x) and x not in (float("inf"), float("-inf")))).sum()
        if bad > 0:
            add("bad_volume_values", f"{bad} NaN/inf in '{col}'", "error")
        neg = (df[col] < 0).sum()
        if neg > 0:
            add("negative_volume", f"{neg} rows with negative '{col}'", "error")

    # 9. Spread bounds
    spread = df["ask"] - df["bid"]
    spread_warn = (spread > SPREAD_MAX_OK).sum()
    spread_err  = (spread > SPREAD_MAX_ERR).sum()
    if spread_err > 0:
        add("spread_too_large", f"{spread_err} rows with spread > {SPREAD_MAX_ERR} ({SPREAD_MAX_ERR*1e4:.0f} pips)", "error")
    elif spread_warn > 0:
        add("spread_elevated", f"{spread_warn} rows with spread > {SPREAD_MAX_OK} ({SPREAD_MAX_OK*1e4:.0f} pips)", "warn")

    # 10. Price bounds
    price_bad = ((df["bid"] < PRICE_MIN) | (df["ask"] > PRICE_MAX)).sum()
    if price_bad > 0:
        add("price_out_of_bounds", f"{price_bad} rows outside [{PRICE_MIN}, {PRICE_MAX}]", "error")

    status = "ok"
    if any(i["severity"] == "error" for i in issues):
        status = "error"
    elif any(i["severity"] == "warn" for i in issues):
        status = "warn"

    return status, warn_codes, issues


In [6]:
dates = pd.date_range(START_DATE, END_DATE, freq="D")

manifest_rows: list[dict] = []
report_rows:   list[dict] = []

for day_ts in dates:
    day_str = day_ts.strftime("%Y-%m-%d")
    parquet_path = OUT_DIR / f"{SYMBOL}_{day_str}_ticks_utc.parquet"

    cal_row      = cal.loc[day_ts] if day_ts in cal.index else None
    is_weekend   = bool(cal_row["is_weekend"])      if cal_row is not None else None
    is_holiday   = bool(cal_row["is_us_holiday"])   if cal_row is not None else None
    is_expected  = bool(cal_row["is_expected_data"]) if cal_row is not None else None

    # â”€â”€ Download & decode â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    df, missing_hours = download_day(SYMBOL, day_str)

    # â”€â”€ Dedup: drop exact full-row duplicates only â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    n_before = len(df)
    df = df.drop_duplicates()
    n_dropped_dupes = n_before - len(df)

    # â”€â”€ Sort â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    df = df.sort_values("timestamp_utc", kind="stable").reset_index(drop=True)

    # â”€â”€ Validation â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    status, warn_codes, day_issues = validate_day(df, day_str, missing_hours, cal)
    report_rows.extend(day_issues)

    # â”€â”€ Save Parquet (even on warn; skip on error with 0 rows) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    if len(df) > 0:
        df.to_parquet(parquet_path, index=False)

    # â”€â”€ Manifest row â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    min_ts = df["timestamp_utc"].min() if len(df) > 0 else None
    max_ts = df["timestamp_utc"].max() if len(df) > 0 else None
    dup_ts = int(df["timestamp_utc"].duplicated().sum())

    manifest_rows.append({
        "date":           day_str,
        "rows":           len(df),
        "min_ts":         str(min_ts) if min_ts is not None else "",
        "max_ts":         str(max_ts) if max_ts is not None else "",
        "missing_hours":  json.dumps(missing_hours),
        "dup_ts_count":   dup_ts,
        "dropped_dupes":  n_dropped_dupes,
        "is_weekend":     is_weekend,
        "is_us_holiday":  is_holiday,
        "is_expected_data": is_expected,
        "status":         status,
        "warnings":       json.dumps(warn_codes),
    })

    icon = {"ok": "[ok]", "warn": "[!]", "error": "[X]"}.get(status, "?")
    print(f"{icon} {day_str}  rows={len(df):>7,}  missing_h={len(missing_hours):>2}  dropped_dupes={n_dropped_dupes}  status={status}")

print("\nIngest loop complete.")


[ok] 2021-03-01  rows= 88,994  missing_h= 0  dropped_dupes=0  status=ok


[ok] 2021-03-02  rows= 91,013  missing_h= 0  dropped_dupes=0  status=ok


[ok] 2021-03-03  rows= 97,956  missing_h= 0  dropped_dupes=0  status=ok


[ok] 2021-03-04  rows=125,449  missing_h= 0  dropped_dupes=0  status=ok


[ok] 2021-03-05  rows=130,957  missing_h= 2  dropped_dupes=0  status=ok


...

[!] 2025-11-04  rows= 53,343  missing_h= 3  dropped_dupes=0  status=warn
[X] 2025-11-05  rows=      0  missing_h=24  dropped_dupes=0  status=error
[X] 2025-11-06  rows=      0  missing_h=24  dropped_dupes=0  status=error
[X] 2025-11-07  rows=      0  missing_h=24  dropped_dupes=0  status=error
[ok] 2025-11-08  rows=      0  missing_h=24  dropped_dupes=0  status=ok
[ok] 2025-11-09  rows=      0  missing_h=24  dropped_dupes=0  status=ok
[X] 2025-11-10  rows=      0  missing_h=24  dropped_dupes=0  status=error
[ok] 2025-11-11  rows=      0  missing_h=24  dropped_dupes=0  status=ok
[X] 2025-11-12  rows=      0  missing_h=24  dropped_dupes=0  status=error
[X] 2025-11-13  rows=      0  missing_h=24  dropped_dupes=0  status=error
[X] 2025-11-14  rows=      0  missing_h=24  dropped_dupes=0  status=error
[ok] 2025-11-15  rows=      0  missing_h=24  dropped_dupes=0  status=ok
[ok] 2025-11-16  rows=      0  missing_h=24  dropped_dupes=0  status=ok
[X] 2025-11-17  rows=      0  missing_h=24  dropp

[ok] 2025-12-14  rows=      0  missing_h=24  dropped_dupes=0  status=ok
[X] 2025-12-15  rows=      0  missing_h=24  dropped_dupes=0  status=error
[X] 2025-12-16  rows=      0  missing_h=24  dropped_dupes=0  status=error
[X] 2025-12-17  rows=      0  missing_h=24  dropped_dupes=0  status=error
[X] 2025-12-18  rows=      0  missing_h=24  dropped_dupes=0  status=error


[X] 2025-12-19  rows=      0  missing_h=24  dropped_dupes=0  status=error
[ok] 2025-12-20  rows=      0  missing_h=24  dropped_dupes=0  status=ok
[ok] 2025-12-21  rows=      0  missing_h=24  dropped_dupes=0  status=ok
[X] 2025-12-22  rows=      0  missing_h=24  dropped_dupes=0  status=error
[X] 2025-12-23  rows=      0  missing_h=24  dropped_dupes=0  status=error
[X] 2025-12-24  rows=      0  missing_h=24  dropped_dupes=0  status=error
[ok] 2025-12-25  rows=      0  missing_h=24  dropped_dupes=0  status=ok
[X] 2025-12-26  rows=      0  missing_h=24  dropped_dupes=0  status=error
[ok] 2025-12-27  rows=      0  missing_h=24  dropped_dupes=0  status=ok
[ok] 2025-12-28  rows=      0  missing_h=24  dropped_dupes=0  status=ok
[X] 2025-12-29  rows=      0  missing_h=24  dropped_dupes=0  status=error
[X] 2025-12-30  rows=      0  missing_h=24  dropped_dupes=0  status=error
[X] 2025-12-31  rows=      0  missing_h=24  dropped_dupes=0  status=error
[ok] 2026-01-01  rows=      0  missing_h=24  dro

[X] 2026-01-22  rows=      0  missing_h=24  dropped_dupes=0  status=error
[X] 2026-01-23  rows=      0  missing_h=24  dropped_dupes=0  status=error
[ok] 2026-01-24  rows=      0  missing_h=24  dropped_dupes=0  status=ok
[ok] 2026-01-25  rows=      0  missing_h=24  dropped_dupes=0  status=ok
[X] 2026-01-26  rows=      0  missing_h=24  dropped_dupes=0  status=error
[X] 2026-01-27  rows=      0  missing_h=24  dropped_dupes=0  status=error
[X] 2026-01-28  rows=      0  missing_h=24  dropped_dupes=0  status=error
[X] 2026-01-29  rows=      0  missing_h=24  dropped_dupes=0  status=error
[X] 2026-01-30  rows=      0  missing_h=24  dropped_dupes=0  status=error
[ok] 2026-01-31  rows=      0  missing_h=24  dropped_dupes=0  status=ok
[ok] 2026-02-01  rows=      0  missing_h=24  dropped_dupes=0  status=ok
[X] 2026-02-02  rows=      0  missing_h=24  dropped_dupes=0  status=error
[X] 2026-02-03  rows=      0  missing_h=24  dropped_dupes=0  status=error
[X] 2026-02-04  rows=      0  missing_h=24  dr

[X] 2026-02-23  rows=      0  missing_h=24  dropped_dupes=0  status=error
[X] 2026-02-24  rows=      0  missing_h=24  dropped_dupes=0  status=error
[X] 2026-02-25  rows=      0  missing_h=24  dropped_dupes=0  status=error
[X] 2026-02-26  rows=      0  missing_h=24  dropped_dupes=0  status=error
[X] 2026-02-27  rows=      0  missing_h=24  dropped_dupes=0  status=error
[ok] 2026-02-28  rows=      0  missing_h=24  dropped_dupes=0  status=ok
[ok] 2026-03-01  rows=      0  missing_h=24  dropped_dupes=0  status=ok
[X] 2026-03-02  rows=      0  missing_h=24  dropped_dupes=0  status=error
[X] 2026-03-03  rows=      0  missing_h=24  dropped_dupes=0  status=error
[X] 2026-03-04  rows=      0  missing_h=24  dropped_dupes=0  status=error
[X] 2026-03-05  rows=      0  missing_h=24  dropped_dupes=0  status=error
[X] 2026-03-06  rows=      0  missing_h=24  dropped_dupes=0  status=error
[ok] 2026-03-07  rows=      0  missing_h=24  dropped_dupes=0  status=ok
[ok] 2026-03-08  rows=      0  missing_h=24 

In [7]:
manifest = pd.DataFrame(manifest_rows)
report   = pd.DataFrame(report_rows) if report_rows else pd.DataFrame(
    columns=["date", "issue", "detail", "severity"]
)

# Append to existing files if they exist (for incremental runs)
if MANIFEST_PATH.exists():
    existing = pd.read_csv(MANIFEST_PATH, dtype=str)
    manifest = pd.concat([existing, manifest], ignore_index=True).drop_duplicates(subset="date", keep="last")

if REPORT_PATH.exists():
    existing_report = pd.read_csv(REPORT_PATH, dtype=str)
    report = pd.concat([existing_report, report], ignore_index=True)
    report = report.drop_duplicates(subset=["date", "issue"], keep="last")

manifest.sort_values("date").to_csv(MANIFEST_PATH, index=False)
report.sort_values("date").to_csv(REPORT_PATH, index=False)

print(f"Manifest saved â†’ {MANIFEST_PATH}  ({len(manifest)} rows)")
print(f"Report saved   â†’ {REPORT_PATH}  ({len(report)} rows)")


Manifest saved â†’ C:\Users\user\EUR_USD_Project\data\dirty\ticks\raw_source\dukascopy\manifest.csv  (1853 rows)
Report saved   â†’ C:\Users\user\EUR_USD_Project\data\dirty\ticks\raw_source\dukascopy\validation_report.csv  (217 rows)


In [8]:
# Merge daily Parquet -> one file; verify row count; delete dailies
if not MERGE_TO_SINGLE_PARQUET:
    print("MERGE_TO_SINGLE_PARQUET is False -- skipping merge.")
else:
    daily_pattern = re.compile(r"^EURUSD_(\d{4}-\d{2}-\d{2})_ticks_utc\.parquet$")
    daily_files = sorted(
        (f for f in OUT_DIR.iterdir() if f.is_file() and daily_pattern.match(f.name)),
        key=lambda p: daily_pattern.match(p.name).group(1),
    )
    if not daily_files:
        raise RuntimeError("No daily Parquet files found to merge.")

    tag_s = START_DATE.replace("-", "")
    tag_e = END_DATE.replace("-", "")
    COMBINED_PATH = OUT_DIR / f"EURUSD_ticks_utc_combined_{tag_s}_{tag_e}.parquet"

    if COMBINED_PATH.exists():
        COMBINED_PATH.unlink()

    writer = None
    for fpath in daily_files:
        tbl = pq.read_table(fpath)
        if writer is None:
            writer = pq.ParquetWriter(str(COMBINED_PATH), tbl.schema, compression="snappy")
        writer.write_table(tbl)
    writer.close()

    combined_rows = pq.ParquetFile(COMBINED_PATH).metadata.num_rows

    m = pd.read_csv(MANIFEST_PATH)
    m["rows"] = pd.to_numeric(m["rows"], errors="coerce").fillna(0).astype(int)
    dates_in_files = {daily_pattern.match(f.name).group(1) for f in daily_files}
    expected = int(m.loc[m["date"].isin(dates_in_files), "rows"].sum())

    print(f"Combined file : {COMBINED_PATH}")
    print(f"Combined rows : {combined_rows:,}")
    print(f"Manifest sum  : {expected:,}")
    if combined_rows != expected:
        raise RuntimeError(f"Row count mismatch: combined={combined_rows} vs manifest={expected}")

    if DELETE_DAILY_AFTER_MERGE:
        for fpath in daily_files:
            fpath.unlink()
        print(f"Deleted {len(daily_files)} daily Parquet file(s).")
    else:
        print("DELETE_DAILY_AFTER_MERGE is False -- daily files kept.")


Combined file : C:\Users\user\EUR_USD_Project\data\dirty\ticks\raw_source\dukascopy\EURUSD_ticks_utc_combined_20210301_20260327.parquet
Combined rows : 118,993,994
Manifest sum  : 118,993,994


Deleted 1461 daily Parquet file(s).


In [9]:
ok   = (manifest["status"] == "ok").sum()
warn = (manifest["status"] == "warn").sum()
err  = (manifest["status"] == "error").sum()
total_rows = pd.to_numeric(manifest["rows"], errors="coerce").sum()

print("=== Ingest Summary ===")
print(f"Days processed : {len(manifest)}")
print(f"  âœ“ ok         : {ok}")
print(f"  âš  warn       : {warn}")
print(f"  âœ— error      : {err}")
print(f"Total ticks    : {int(total_rows):,}")

if warn + err > 0:
    print("\nDays needing attention:")
    display(manifest[manifest["status"] != "ok"][["date", "rows", "missing_hours", "status", "warnings"]])

print("\nFull manifest:")
display(manifest[["date", "rows", "missing_hours", "dup_ts_count", "is_weekend", "is_us_holiday", "status"]])


=== Ingest Summary ===
Days processed : 1853
  âœ“ ok         : 1734
  âš  warn       : 15
  âœ— error      : 104
Total ticks    : 118,993,994

Days needing attention:


,date,rows,missing_hours,status,warnings
101,2021-05-10,57127,"[16, 22, 23]",warn,"[""missing_hours""]"
132,2021-06-10,64644,"[21, 22, 23]",warn,"[""missing_hours""]"
133,2021-06-11,0,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...",error,"[""empty_expected_day"", ""missing_hours""]"
420,2022-03-25,65766,"[15, 16, 17, 18, 19, 20, 21, 22, 23]",error,"[""missing_hours""]"
451,2022-04-25,57704,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 15]",error,"[""missing_hours""]"
...,...,...,...,...,...
1879,2026-03-23,0,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...",error,"[""empty_expected_day"", ""missing_hours""]"
1880,2026-03-24,0,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...",error,"[""empty_expected_day"", ""missing_hours""]"
1881,2026-03-25,0,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...",error,"[""empty_expected_day"", ""missing_hours""]"
1882,2026-03-26,0,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...",error,"[""empty_expected_day"", ""missing_hours""]"



Full manifest:


,date,rows,missing_hours,dup_ts_count,is_weekend,is_us_holiday,status
31,2021-03-01,88994,[],0,False,False,ok
32,2021-03-02,91013,[],0,False,False,ok
33,2021-03-03,97956,[],0,False,False,ok
34,2021-03-04,125449,[],0,False,False,ok
35,2021-03-05,130957,"[22, 23]",0,False,False,ok
...,...,...,...,...,...,...,...
1879,2026-03-23,0,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...",0,False,False,error
1880,2026-03-24,0,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...",0,False,False,error
1881,2026-03-25,0,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...",0,False,False,error
1882,2026-03-26,0,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...",0,False,False,error
